#  BiLSTM Leave-One Participant Out Prediction (15 minutes before the event)



In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

## 1. Configuration

Set paths and model parameters.

In [2]:
DATA_ROOT = Path("../R21_data")  # base of the synthetic data
PRE_WINDOW_MIN = 15  # minutes before a smoking event that the features cover
WINDOW_LEN = 17  # consecutive rows grouped into one LSTM sequence
EPOCHS = 30  # base-model training epochs
FINETUNE_EPOCHS = 1  # a light training on the held-out participant
BATCH_SIZE = 32
FINETUNE_BATCH_SIZE = 30
RANDOM_SEED = 9  # fixed seed so splits and weight init are reproducible


## 2. Model

A compact two-layer bidirectional LSTM binary classifier.

In [3]:
def bi_lstm_model(input_shape: tuple[int, int]) -> keras.Model:
    """Build and compile a two-layer bidirectional LSTM classifier.

    Args:
        input_shape: Tuple of ``(time_steps, n_features)`` for each input sequence.

    Returns:
        A compiled Keras model with a single sigmoid output.
    """
    inputs = keras.Input(shape=input_shape, dtype="float32")
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(inputs)  # keep full sequence to feed the next LSTM
    x = layers.Bidirectional(layers.LSTM(64))(x)  # collapse the sequence to a single summary vector
    outputs = layers.Dense(1, activation="sigmoid")(x)  # sigmoid -> probability of the positive (smoking) class
    model = keras.Model(inputs, outputs)
    model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])  
    return model


## 3. Data loading and reshaping

Load the combined pre-15 feature table, then reshape the flat rows into fixed-length
windowed sequences grouped by smoking event.

In [4]:
def _reshape_windows(
    df: pd.DataFrame,
    feature_cols: list[str],
    window_len: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Reshape flat feature rows into fixed-length windowed sequences.

    Args:
        df: Feature dataframe containing ``smoking_label`` and ``smoking_index``.
        feature_cols: Ordered list of feature column names.
        window_len: Number of consecutive rows that make up one window.

    Returns:
        A tuple ``(X, y, groups)`` where ``X`` has shape
        ``(n_windows, window_len, n_features)``, ``y`` holds one binary label per
        window, and ``groups`` holds the smoking-event id per window.
    """
    X_flat = df[feature_cols].to_numpy(dtype=np.float32)  # feature matrix, one row per time step
    y_flat = df["smoking_label"].to_numpy(dtype=np.float32)  # per-row smoking label
    group_flat = df["smoking_index"].to_numpy()  # per-row event id (keeps an event in one CV fold)

    usable = (len(df) // window_len) * window_len  # trim the remainder so rows divide evenly into windows
    X_flat = X_flat[:usable]
    y_flat = y_flat[:usable]
    group_flat = group_flat[:usable]

    X = X_flat.reshape(-1, window_len, len(feature_cols))  # (n_windows, window_len, n_features)
    y = y_flat[::window_len]  # one label per window, taken from its first row
    groups = group_flat[::window_len]  # one event id per window, used for grouped CV
    return X, y, groups


def _load_all_features() -> pd.DataFrame:
    """Load the combined pre-15 feature table and drop non-feature columns.

    Returns:
        A dataframe with metadata columns removed and the index reset.
    """
    path = (
        DATA_ROOT
        / "features_around_events"
        / "H10_features"
        / f"h10_features_everyone_pre{PRE_WINDOW_MIN}_clean.pkl"
    )
    df = pd.read_pickle(path)
    drop_cols = [
        col
        for col in ["locationname", "timestamp", "longitude", "latitude", "source_file"]
        if col in df.columns
    ]  # metadata columns that must not be fed to the model as features
    if drop_cols:
        df = df.drop(columns=drop_cols)
    return df.reset_index(drop=True)

## 4. Leave-one-participant-out precision evaluation

Train a base model on every other participant, fine-tune on a small slice of the held-out
participant, then score precision on the rest of that participant's windows.

In [5]:
def run_lopo_precision() -> pd.DataFrame:
    """Run leave-one-participant-out precision evaluation.

    For each participant the model is trained on everyone else (with a
    ``StratifiedGroupKFold`` outer split), briefly fine-tuned on a slice of the
    held-out participant, and then scored. A per-participant summary is written to
    disk and returned.

    Returns:
        A dataframe with one row per scored participant containing base accuracy,
        fine-tuned accuracy and the last confusion matrix.

    Raises:
        ValueError: If required columns are missing from the feature table.
    """
    tf.random.set_seed(RANDOM_SEED)  # reproducible weight init and shuffling

    all_df = _load_all_features()
    required = {"p_id", "smoking_index", "smoking_label"}
    missing = required - set(all_df.columns)
    if missing:  # fail fast if the feature table is missing id/label columns
        raise ValueError(f"Missing required columns in features dataframe: {sorted(missing)}")

    feature_cols = [
        c for c in all_df.columns if c not in {"p_id", "smoking_index", "smoking_label"}
    ]  # everything except id/label columns is a model feature
    scaler = StandardScaler(copy=False)
    all_df.loc[:, feature_cols] = scaler.fit_transform(all_df.loc[:, feature_cols].to_numpy())  # standardize features in place

    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,  # roll back to the best epoch when val loss stops improving
        verbose=0,
    )

    result_rows: list[dict[str, object]] = []
    participant_ids = sorted(all_df["p_id"].dropna().unique().tolist())

    for pid in participant_ids:  # leave-one-participant-out: hold out each participant in turn
        test_df = all_df[all_df["p_id"] == pid].reset_index(drop=True)  # the held-out participant
        train_df = all_df[all_df["p_id"] != pid].reset_index(drop=True)  # everyone else
        if len(test_df) < WINDOW_LEN or len(train_df) < WINDOW_LEN:
            continue  # not enough rows to build even one window

        X_test, y_test, groups_test = _reshape_windows(test_df, feature_cols, WINDOW_LEN)
        X_train_full, y_train_full, groups_train_full = _reshape_windows(
            train_df, feature_cols, WINDOW_LEN
        )

        cv_outer = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=RANDOM_SEED)  # group-aware so an event never straddles train/val
        outer_scores: list[float] = []
        finetune_scores: list[float] = []
        cms: list[np.ndarray] = []

        for train_idx, val_idx in cv_outer.split(X_train_full, y_train_full, groups_train_full):
            X_train, X_val = X_train_full[train_idx], X_train_full[val_idx]
            y_train, y_val = y_train_full[train_idx], y_train_full[val_idx]

            model = bi_lstm_model((X_train.shape[1], X_train.shape[2]))  # fresh model for each outer fold
            model.fit(
                X_train,
                y_train,
                epochs=EPOCHS,
                batch_size=BATCH_SIZE,
                validation_data=(X_val, y_val),
                callbacks=[early_stop],
                verbose=0,
                shuffle=False,  # preserve temporal order of the sequences
            )

            _, base_acc = model.evaluate(X_test, y_test, verbose=0)  # accuracy on the held-out participant before fine-tuning
            outer_scores.append(float(base_acc))

            cv_inner = StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=RANDOM_SEED)  # split the held-out participant: fine-tune vs score
            for tv_idx, test_idx in cv_inner.split(X_test, y_test, groups_test):
                X_tv, X_tune_test = X_test[tv_idx], X_test[test_idx]  # tv = fine-tune pool, tune_test = scored portion
                y_tv, y_tune_test = y_test[tv_idx], y_test[test_idx]
                g_tv = groups_test[tv_idx]

                for ft_train_idx, ft_val_idx in cv_inner.split(X_tv, y_tv, g_tv):  # further split the fine-tune pool into train/val
                    X_ft_train, X_ft_val = X_tv[ft_train_idx], X_tv[ft_val_idx]
                    y_ft_train, y_ft_val = y_tv[ft_train_idx], y_tv[ft_val_idx]

                    model.fit(
                        X_ft_train,
                        y_ft_train,
                        epochs=FINETUNE_EPOCHS,
                        batch_size=FINETUNE_BATCH_SIZE,
                        validation_data=(X_ft_val, y_ft_val),
                        callbacks=[early_stop],
                        verbose=0,
                        shuffle=False,
                    )  # brief fine-tune to adapt the base model to this participant

                _, tune_acc = model.evaluate(X_tune_test, y_tune_test, verbose=0)  # accuracy after fine-tuning
                finetune_scores.append(float(tune_acc))
                preds = (model.predict(X_tune_test, verbose=0).reshape(-1) >= 0.5).astype(int)  # threshold probabilities at 0.5
                cms.append(confusion_matrix(y_tune_test.astype(int), preds))

        if not outer_scores:
            continue  # nothing scored for this participant

        result_rows.append(
            {
                "p_id": pid,
                "train_accuracy": float(np.mean(outer_scores)),  # mean base accuracy across outer folds
                "finetune_accuracy": float(np.mean(finetune_scores)) if finetune_scores else np.nan,  # mean post-fine-tune accuracy
                "n_confusion_matrices": len(cms),
                "cm_last": cms[-1].tolist() if cms else None,  # keep the last confusion matrix for inspection
            }
        )

    summary = pd.DataFrame(result_rows)
    out_dir = DATA_ROOT / "ml_models" / "bilstm_precision_pre15_clean"
    out_dir.mkdir(parents=True, exist_ok=True)
    summary.to_csv(out_dir / "bilstm_pre15_lopo_summary.csv", index=False)

    if not summary.empty:
        agg = pd.DataFrame(
            {
                "metric": ["train_mean", "train_std", "finetune_mean", "finetune_std"],
                "value": [
                    summary["train_accuracy"].mean(),
                    summary["train_accuracy"].std(),
                    summary["finetune_accuracy"].mean(),
                    summary["finetune_accuracy"].std(),
                ],
            }
        )  # mean/std of accuracy across participants
        agg.to_csv(out_dir / "bilstm_pre15_lopo_aggregate.csv", index=False)

    return summary

## 5. Run

Execute the full LOPO precision evaluation and display the per-participant summary.

In [6]:
summary_df = run_lopo_precision()
summary_df

,p_id,train_accuracy,finetune_accuracy,n_confusion_matrices,cm_last
0,sample01,0.952381,0.964773,8,"[[5, 0], [0, 5]]"
1,sample02,0.952381,0.988636,8,"[[5, 0], [0, 5]]"
2,sample03,0.964286,0.976136,8,"[[5, 0], [0, 5]]"
3,sample04,0.964286,0.976136,8,"[[5, 0], [0, 5]]"
4,sample05,0.952381,0.964773,8,"[[5, 0], [0, 5]]"
